# 🔄 Notebook 3: Resumable Uploads

For large files, use multipart uploads so failures don't mean starting over.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why resumable uploads matter
- Multipart upload API
- Tracking upload progress
- Handling failures gracefully

In [ ]:
import boto3
from botocore.config import Config
import os
import hashlib
from typing import List, Dict
import random

s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

BUCKET = 'uploads'

print("✅ Connected to MinIO!")
print("📊 Open MinIO Console: http://localhost:9001")

## 🔄 Why Resumable Uploads?

In [ ]:
print("🔄 Why Resumable Uploads Matter")
print("=" * 60)
print("""
SCENARIO: Uploading a 5GB video on mobile
─────────────────────────────────────────────────────────────

At 50 Mbps, upload takes ~14 minutes.

WITHOUT RESUMABLE:
┌─────────────────────────────────────────────────────────┐
│ Progress: [████████████████████████████████████░░] 99% │
│                                                         │
│ ❌ NETWORK ERROR!                                       │
│                                                         │
│ "Your upload failed. Please try again."                │
│                                                         │
│ Progress: [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 0%  │
└─────────────────────────────────────────────────────────┘
                    START OVER! 😱

─────────────────────────────────────────────────────────────

WITH RESUMABLE (Multipart):
┌─────────────────────────────────────────────────────────┐
│ Progress: [████████████████████████████████████░░] 99% │
│ Parts: 99/100 complete                                  │
│                                                         │
│ ⚠️ NETWORK ERROR!                                       │
│                                                         │
│ "Resuming upload..."                                    │
│                                                         │
│ Progress: [████████████████████████████████████░░] 99% │
│ Resuming from part 100...                               │
└─────────────────────────────────────────────────────────┘
                    RESUME FROM FAILURE! 🎉
""")

## 📦 Multipart Upload API

In [ ]:
print("📦 How Multipart Upload Works")
print("=" * 60)
print("""
STEP 1: Initiate Multipart Upload
─────────────────────────────────────────────────────────────
Client → S3: "I want to upload large-video.mp4"
S3 → Client: "Here's your upload_id: abc123"

STEP 2: Upload Parts (can be parallel!)
─────────────────────────────────────────────────────────────
┌─────────────────────────────────────────────────────────┐
│  Part 1    Part 2    Part 3    Part 4    Part 5        │
│  [5MB]     [5MB]     [5MB]     [5MB]     [3MB]         │
│    │         │         │         │         │           │
│    ▼         ▼         ▼         ▼         ▼           │
│  ETag:     ETag:     ETag:     ETag:     ETag:         │
│  "aaa"     "bbb"     "ccc"     "ddd"     "eee"         │
└─────────────────────────────────────────────────────────┘

Each part returns an ETag (checksum) for verification.

STEP 3: Complete Upload
─────────────────────────────────────────────────────────────
Client → S3: "Here are all parts and their ETags"
S3: Assembles parts into final object

PART SIZE REQUIREMENTS:
• Minimum: 5MB (except last part)
• Maximum: 5GB per part
• Maximum: 10,000 parts
""")

In [ ]:
class MultipartUploader:
    def __init__(self, bucket: str, key: str, part_size: int = 5 * 1024 * 1024):
        self.bucket = bucket
        self.key = key
        self.part_size = part_size
        self.upload_id = None
        self.parts: List[Dict] = []
    
    def initiate(self) -> str:
        response = s3.create_multipart_upload(
            Bucket=self.bucket,
            Key=self.key
        )
        self.upload_id = response['UploadId']
        return self.upload_id
    
    def upload_part(self, part_number: int, data: bytes) -> dict:
        response = s3.upload_part(
            Bucket=self.bucket,
            Key=self.key,
            UploadId=self.upload_id,
            PartNumber=part_number,
            Body=data
        )
        
        part_info = {
            'PartNumber': part_number,
            'ETag': response['ETag']
        }
        self.parts.append(part_info)
        return part_info
    
    def complete(self) -> dict:
        sorted_parts = sorted(self.parts, key=lambda x: x['PartNumber'])
        
        response = s3.complete_multipart_upload(
            Bucket=self.bucket,
            Key=self.key,
            UploadId=self.upload_id,
            MultipartUpload={'Parts': sorted_parts}
        )
        return response
    
    def abort(self):
        s3.abort_multipart_upload(
            Bucket=self.bucket,
            Key=self.key,
            UploadId=self.upload_id
        )
    
    def list_uploaded_parts(self) -> List[dict]:
        response = s3.list_parts(
            Bucket=self.bucket,
            Key=self.key,
            UploadId=self.upload_id
        )
        return response.get('Parts', [])

print("✅ MultipartUploader class defined!")

In [ ]:
print("📤 Multipart Upload Demo")
print("=" * 60)

total_size = 25 * 1024 * 1024
part_size = 5 * 1024 * 1024
num_parts = (total_size + part_size - 1) // part_size

print(f"\n📊 Upload Configuration:")
print(f"   Total file size: {total_size / 1024 / 1024:.0f}MB")
print(f"   Part size: {part_size / 1024 / 1024:.0f}MB")
print(f"   Number of parts: {num_parts}")

uploader = MultipartUploader(BUCKET, 'videos/large-video.mp4', part_size)

print("\n1️⃣ Initiating multipart upload...")
upload_id = uploader.initiate()
print(f"   Upload ID: {upload_id[:20]}...")

print("\n2️⃣ Uploading parts...")
for part_num in range(1, num_parts + 1):
    if part_num == num_parts:
        remaining = total_size - (part_size * (num_parts - 1))
        data = os.urandom(remaining)
    else:
        data = os.urandom(part_size)
    
    part_info = uploader.upload_part(part_num, data)
    progress = part_num / num_parts * 100
    bar = "█" * int(progress / 5) + "░" * (20 - int(progress / 5))
    print(f"   Part {part_num}/{num_parts}: [{bar}] {progress:.0f}% - ETag: {part_info['ETag'][:10]}...")

print("\n3️⃣ Completing upload...")
result = uploader.complete()
print(f"   ✅ Upload complete!")
print(f"   Location: {result.get('Location', 'N/A')}")

## 🔁 Simulating Resume After Failure

In [ ]:
print("🔁 Simulating Upload Failure and Resume")
print("=" * 60)

uploader2 = MultipartUploader(BUCKET, 'videos/resume-demo.mp4', 5 * 1024 * 1024)

print("\n1️⃣ Starting upload (will 'fail' at part 3)...")
upload_id = uploader2.initiate()
print(f"   Upload ID: {upload_id[:20]}...")

for part_num in range(1, 6):
    data = os.urandom(5 * 1024 * 1024)
    
    if part_num == 3:
        print(f"   Part {part_num}: ❌ NETWORK ERROR! (simulated)")
        break
    
    uploader2.upload_part(part_num, data)
    print(f"   Part {part_num}: ✅ Uploaded")

print("\n📋 Checking which parts were uploaded...")
existing_parts = uploader2.list_uploaded_parts()
uploaded_part_numbers = {p['PartNumber'] for p in existing_parts}
print(f"   Parts already uploaded: {sorted(uploaded_part_numbers)}")

print("\n2️⃣ Resuming from where we left off...")
for part_num in range(1, 6):
    if part_num in uploaded_part_numbers:
        print(f"   Part {part_num}: ⏭️ Already uploaded, skipping")
        continue
    
    data = os.urandom(5 * 1024 * 1024)
    uploader2.upload_part(part_num, data)
    print(f"   Part {part_num}: ✅ Uploaded")

print("\n3️⃣ Completing...")
uploader2.complete()
print("   ✅ Upload completed after resume!")

## 📊 Progress Tracking

In [ ]:
class ProgressTracker:
    def __init__(self, total_parts: int, total_bytes: int):
        self.total_parts = total_parts
        self.total_bytes = total_bytes
        self.completed_parts = 0
        self.completed_bytes = 0
    
    def update(self, part_number: int, bytes_uploaded: int):
        self.completed_parts += 1
        self.completed_bytes += bytes_uploaded
    
    def get_progress(self) -> dict:
        return {
            'parts_completed': self.completed_parts,
            'parts_total': self.total_parts,
            'bytes_completed': self.completed_bytes,
            'bytes_total': self.total_bytes,
            'percent': (self.completed_bytes / self.total_bytes) * 100
        }
    
    def display(self):
        p = self.get_progress()
        bar_len = 30
        filled = int(bar_len * p['percent'] / 100)
        bar = "█" * filled + "░" * (bar_len - filled)
        mb_done = p['bytes_completed'] / 1024 / 1024
        mb_total = p['bytes_total'] / 1024 / 1024
        print(f"   [{bar}] {p['percent']:.1f}% ({mb_done:.1f}/{mb_total:.1f}MB)")

print("📊 Upload with Progress Tracking")
print("=" * 60)

total_size = 30 * 1024 * 1024
part_size = 5 * 1024 * 1024
num_parts = (total_size + part_size - 1) // part_size

uploader3 = MultipartUploader(BUCKET, 'videos/tracked-upload.mp4', part_size)
tracker = ProgressTracker(num_parts, total_size)

uploader3.initiate()
print("\n📤 Uploading with live progress:")

for part_num in range(1, num_parts + 1):
    if part_num == num_parts:
        remaining = total_size - (part_size * (num_parts - 1))
        data = os.urandom(remaining)
        size = remaining
    else:
        data = os.urandom(part_size)
        size = part_size
    
    uploader3.upload_part(part_num, data)
    tracker.update(part_num, size)
    tracker.display()

uploader3.complete()
print("\n✅ Upload complete!")

## 🧹 Cleaning Up Abandoned Multipart Uploads

When a client starts a multipart upload and never completes (or aborts) it,
the parts already uploaded **stay in the bucket and cost money** until someone
cleans them up.

Two ways to deal with this:

1. **Bucket lifecycle rule (recommended in production)** — tell S3/MinIO to
   automatically abort incomplete multipart uploads older than N days.
2. **Manual scan and abort** — list in-progress uploads and abort the ones
   older than your threshold. Useful when you want full control or your
   storage doesn't support lifecycle rules.

Below we demonstrate the manual approach because it shows what's actually
happening under the hood.


In [ ]:
from datetime import datetime, timedelta, timezone

# Start an upload and upload one part, but NEVER complete it.
abandoned = MultipartUploader(BUCKET, 'videos/abandoned.mp4', 5 * 1024 * 1024)
abandoned.initiate()
abandoned.upload_part(1, os.urandom(5 * 1024 * 1024))
print(f"😶 Abandoned upload started with id: {abandoned.upload_id[:20]}...")
print("   (We're deliberately NOT calling complete() or abort())")

# Now, as a cleanup job, list all in-progress multipart uploads.
print("\n🔍 Scanning for in-progress multipart uploads...")
response = s3.list_multipart_uploads(Bucket=BUCKET)
uploads = response.get('Uploads', [])
print(f"   Found {len(uploads)} in-progress upload(s)")

# Abort anything older than our threshold. For demo, abort everything found.
# In production, compare `u['Initiated']` to `now - max_age`.
MAX_AGE = timedelta(hours=24)
now = datetime.now(timezone.utc)

aborted = 0
for u in uploads:
    age = now - u['Initiated']
    too_old = age > MAX_AGE
    marker = "🗑️ abort" if too_old or True else "✅ keep"  # demo: abort all
    print(f"   {marker}  key={u['Key']:30} age={age} id={u['UploadId'][:16]}...")
    s3.abort_multipart_upload(
        Bucket=BUCKET, Key=u['Key'], UploadId=u['UploadId']
    )
    aborted += 1

print(f"\n✅ Aborted {aborted} stale upload(s). Their parts were deleted.")
print("\n💡 In production, use a bucket lifecycle rule instead:")
print("   AbortIncompleteMultipartUpload: DaysAfterInitiation: 7")


## 🧪 Quick Quiz

1. **What's the minimum part size for multipart uploads?**

2. **How do you know which parts to resume after failure?**

3. **What happens if you don't complete or abort a multipart upload?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Minimum part size:")
print("   - 5MB (except for the last part)")
print("   - Maximum: 5GB per part")
print("   - Maximum: 10,000 parts total")
print()
print("2. How to know which parts to resume:")
print("   - Call ListParts with upload_id")
print("   - Returns list of completed parts")
print("   - Upload only missing parts")
print()
print("3. Incomplete multipart uploads:")
print("   - Parts stay in storage, costing money!")
print("   - Set lifecycle rules to auto-delete")
print("   - Usually 1-7 days after last activity")

## 📚 Summary

### Key Takeaways

1. **Multipart = resumable** - Don't start over on failure
2. **Three steps** - Initiate, upload parts, complete
3. **ETags for verification** - Each part has checksum
4. **ListParts for resume** - Know what's already uploaded
5. **Clean up!** - Incomplete uploads cost money

### Next Up

In **Notebook 4**, we'll learn about state synchronization:
- Keeping database and storage in sync
- Event notifications
- Handling edge cases